# Ground-truth address matches

Load the first 10,000 ground-truth rows and retrieve the raw addresses for each Source 1 record and its matched Source 2/3 records. No address cleaning or normalization is applied.


In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('dataset/train')
GROUND_TRUTH_PATH = DATA_DIR / 'train_ground_truth.tsv'
SOURCE1_PATH = DATA_DIR / 'train_source1.tsv'
SOURCE2_PATH = DATA_DIR / 'train_source2.tsv'
SOURCE3_PATH = DATA_DIR / 'train_source3.tsv'

ground_truth = pd.read_csv(
    GROUND_TRUTH_PATH,
    sep='\t',
    dtype=str,
    keep_default_na=False,
    nrows=10_000,
)

print(f'Ground-truth rows loaded: {len(ground_truth):,}')
display(ground_truth.head())


Ground-truth rows loaded: 10,000


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [2]:
# Expand each comma-separated match list into one row per matched entity.
pairs = ground_truth.reset_index(names='ground_truth_index')
pairs['ground_truth_row'] = pairs['ground_truth_index'] + 1
pairs['matched_entity_id'] = pairs['matched_entity_ids'].str.split(',')
pairs = pairs.explode('matched_entity_id')
pairs['matched_entity_id'] = pairs['matched_entity_id'].str.strip()
pairs = pairs[pairs['matched_entity_id'].ne('')].copy()

source1_ids = set(ground_truth['source1_entity_id'])
source2_ids = set(pairs.loc[pairs['matched_entity_id'].str.startswith('S2-'), 'matched_entity_id'])
source3_ids = set(pairs.loc[pairs['matched_entity_id'].str.startswith('S3-'), 'matched_entity_id'])

def retrieve_records(path, wanted_ids, chunksize=250_000):
    records = []
    for chunk in pd.read_csv(
        path, sep='\t', dtype=str, keep_default_na=False, chunksize=chunksize
    ):
        matched = chunk[chunk['entity_id'].isin(wanted_ids)]
        if not matched.empty:
            records.append(matched)
    return pd.concat(records, ignore_index=True)

source1_records = retrieve_records(SOURCE1_PATH, source1_ids)
source2_records = retrieve_records(SOURCE2_PATH, source2_ids)
source3_records = retrieve_records(SOURCE3_PATH, source3_ids)
matched_records = pd.concat([source2_records, source3_records], ignore_index=True)

source1_addresses = source1_records[[
    'entity_id', 'business_address', 'country'
]].rename(columns={
    'entity_id': 'source1_entity_id',
    'business_address': 'source1_address',
    'country': 'source1_country',
})

matched_addresses = matched_records[[
    'entity_id', 'business_address', 'country'
]].rename(columns={
    'entity_id': 'matched_entity_id',
    'business_address': 'matched_address',
    'country': 'matched_country',
})

address_matches = (
    pairs[['ground_truth_row', 'source1_entity_id', 'matched_entity_id']]
    .merge(source1_addresses, on='source1_entity_id', how='left', validate='many_to_one')
    .merge(matched_addresses, on='matched_entity_id', how='left', validate='many_to_one')
)
address_matches['matched_source'] = address_matches['matched_entity_id'].str.slice(0, 2)
address_matches = address_matches[[
    'ground_truth_row',
    'source1_entity_id',
    'source1_address',
    'source1_country',
    'matched_entity_id',
    'matched_source',
    'matched_address',
    'matched_country',
]]

print(f'Matched address rows created: {len(address_matches):,}')
print(f'Missing Source 1 addresses after joining: {address_matches.source1_address.isna().sum():,}')
print(f'Missing matched records after joining: {address_matches.matched_address.isna().sum():,}')


Matched address rows created: 34,511
Missing Source 1 addresses after joining: 0
Missing matched records after joining: 0


In [3]:
# Show the first 100 matched-address rows.
# The complete result for all 10,000 ground-truth rows is stored in address_matches.
display(address_matches.head(100))


,ground_truth_row,source1_entity_id,source1_address,source1_country,matched_entity_id,matched_source,matched_address,matched_country
0,1,S1-965667,"85 Wayne Avenue, Ticonderoga, NY",US,S2-681193310,S2,,US
1,1,S1-965667,"85 Wayne Avenue, Ticonderoga, NY",US,S2-743505751,S2,,US
2,1,S1-965667,"85 Wayne Avenue, Ticonderoga, NY",US,S3-775321672,S3,"85 Wanye Avenue, Ticonderoga Townshiip, New York",US
3,1,S1-965667,"85 Wayne Avenue, Ticonderoga, NY",US,S3-11291185,S3,"Wayne Ave, Ticonderoga Townshiip, New York",US
4,1,S1-965667,"85 Wayne Avenue, Ticonderoga, NY",US,S3-860443364,S3,,US
...,...,...,...,...,...,...,...,...
95,27,S1-292935703,"617 Fourth Street, Watseka, IL",US,S2-195749344,S2,"617 FOURTH ST, WATSEKA, IL",US
96,27,S1-292935703,"617 Fourth Street, Watseka, IL",US,S2-617410789,S2,"4TH STREET, WATSEKA, IL",US
97,27,S1-292935703,"617 Fourth Street, Watseka, IL",US,S2-516200703,S2,"617 4TH ST, WATSEKA, IL",US
98,27,S1-292935703,"617 Fourth Street, Watseka, IL",US,S3-421403180,S3,"Fourth Saint, Watseka, Illinois",US


In [3]:
address_view = (
    address_matches
    .groupby(
        ['ground_truth_row', 'source1_address', 'matched_source'],
        sort=False
    )['matched_address']
    .agg(list)
    .unstack()
    .reset_index()
)

# Ensure both columns exist.
for source in ['S2', 'S3']:
    if source not in address_view.columns:
        address_view[source] = None

address_view = address_view.rename(columns={
    'source1_address': 'source1_addresses',
    'S2': 'source2_addresses',
    'S3': 'source3_addresses',
})

display(
    address_view[
        ['source1_addresses', 'source2_addresses', 'source3_addresses']
    ].head(100)
)

matched_source,source1_addresses,source2_addresses,source3_addresses
0,"85 Wayne Avenue, Ticonderoga, NY","[, ]","[85 Wanye Avenue, Ticonderoga Townshiip, New Y..."
1,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, ...","[6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE,...","[6(29), C.i.t. Colony, 2Nd Main Road Mylapore,..."
2,"630 45th Terrace, Kansas City, MO","[KANSAS CITY, MO, 630 45ND TERRACE, null, 45ND...","[Missouri, 630 45th Terrace, Kansas City]"
3,"Af-684, Nandgram Near Mother India Public Scho...","[AF-0684, NANDGRAM NEAR MOTHER INDIA PUBLIC SC...","[Af-684, Ghaziabad, UP]"
4,"3315 Fremont Street, Peoria, IL","[3315 FREMONT ST, PEORIA, IL, 3315 FREMONT ST,...","[3315 Fremont St, Peoria, Illinois, 3315 Fremo..."
...,...,...,...
95,"18 Day Street, Unit 208, Somerville, MA","[18 DAY STREET, PO BOX 9605, SOMERVILLE, MA, 1...","[18 Day Street, # 208, Somerville Township, Ma..."
96,"#12 Ward No 22, 2Nd Cross, Rahamaniya Nagar, R...","[NO 12 WARD NO 22, 2ND CROSS, RAHAMANIYA NAGAR...","[H.no 012 Ward No 22, 2Nd Cross, Rahamaniya Na..."
97,"129 Geneva Road, New Bern, NC","[129 GENEVA ROAD, NEW BERN, NC, 129 GENEVA ROA...","[129 Geneva Road, New Bern, North Carolina, No..."
98,"14/307, Third Floor, Regency Appartment Johri ...","[##14/307, THIRD FLOOR, REGENCY APPARTMENT JOH...",NaN


In [5]:
import pandas as pd
from IPython.display import display, HTML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Prevent Jupyter from collapsing long output.
display(HTML("""
<style>
.output_scroll,
.jp-OutputArea-output {
    height: auto !important;
    max-height: none !important;
    overflow-y: visible !important;
}
</style>
"""))

full_address_view = address_view[
    ['source1_addresses', 'source2_addresses', 'source3_addresses']
].head(100).copy()

# Put each matching address on a separate line.
for column in ['source2_addresses', 'source3_addresses']:
    full_address_view[column] = full_address_view[column].apply(
        lambda value: '\n'.join(value)
        if isinstance(value, list)
        else ''
    )

display(
    full_address_view.style.set_properties(**{
        'white-space': 'pre-wrap',
        'text-align': 'left',
        'vertical-align': 'top',
        'min-width': '300px',
    })
)

matched_source,source1_addresses,source2_addresses,source3_addresses
0,"85 Wayne Avenue, Ticonderoga, NY",,"85 Wanye Avenue, Ticonderoga Townshiip, New York Wayne Ave, Ticonderoga Townshiip, New York"
1,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu","6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu 6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu","6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, TN 6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, தமிழ்நாடு"
2,"630 45th Terrace, Kansas City, MO","KANSAS CITY, MO, 630 45ND TERRACE, null 45ND TERRACE, null, KANSAS CITY, MO","Missouri, 630 45th Terrace, Kansas City"
3,"Af-684, Nandgram Near Mother India Public School. Ph. 989, 9487203, Ghaziabad, Uttar Pradesh","AF-0684, NANDGRAM NEAR MOTHER INDIA PUBLIC SCHOOL. PH. 989, GHAZIABAD, 9487203, उत्तर प्रदेश AF-0684, Uttar Pradesh, GHAZIABAD, 9487203","Af-684, Ghaziabad, UP"
4,"3315 Fremont Street, Peoria, IL","3315 FREMONT ST, PEORIA, IL 3315 FREMONT ST, PEORIA, IL 3315 FREMONT SAINT, PEORIA, IL","3315 Fremont St, Peoria, Illinois 3315 Fremont Street, Peoria, Illinois Fremont St, Peoria, Illinois"
5,"1056 Belden Avenue, Akron, OH","1056-1060 BELDEN AVE, PO BOX 8807, AKRON, OH","1056c Belden Ave, AKON, Ohio 1056c Belden Ave, AKON, Ohio 1056c Belden Avenue, AKON, Ohio"
6,"Rajasthan, Jaipur, Banipark, Gokul Apartment, E-3A Kanti Chandra Road, G-1","G-1, BANIPARK, JAIPUR, Rajasthan","Doro No 316 G-1, Gokul Apartment, E-3a Kanti Chandra Road, Banipark, Subhash Nagar, RJ"
7,"New Bridge Business Centre'S 11Th Floor, N1 Block Embassy Manyata Business Tech Park, Naga, Wara, Bangalore, Karnataka",,"New Bridge Bssiness Centre's 11Th Floor, N1 Block Embassy Manyata Business Tech Park, Naga, Wara, Bangalore, ಕರ್ನಾಟಕ New Bridge Buisness Centre's 11Th Floor, N1 Block Embassy Manyata Business Tech Park, Naga, Wara, Bangalore, KA"
8,"33 Sleepy Hollow Drive, Danbury, CT","CT, SLEEPY HOLLOW DRIVE, DANBURY",
9,"Wz-187C Shop No.13, 14 Kh. No.47 S/F. Vikaspuri Budhela Village Behind Oxford School, Delhi, West Delhi, Delhi","WZ-187C SHOP NO.13, DELHI, WEST DELHI, Delhi","Block B-517 Wz-187c Shop No.13, Divreportingcircle, West Delhi, DL Block B-517 Wz-187c Shop No.13, South West Delhi, Delhi, DL"


In [20]:
BASE_TRAIN = '/home/laalenthika/Downloads/student_resource/dataset/train/'

SOURCE1_PATH = BASE_TRAIN + 'train_source1.tsv'
SOURCE2_PATH = BASE_TRAIN + 'train_source2.tsv'
SOURCE3_PATH = BASE_TRAIN + 'train_source3.tsv'
OUTPUT_DIR = "/home/laalenthika/Downloads/student_resource/output/"

candidate_details = pd.read_csv(OUTPUT_DIR + 'development_candidate_details.tsv', sep='\t', dtype=str, keep_default_na=False)
source1 = pd.read_csv(SOURCE1_PATH, sep='\t', dtype=str, keep_default_na=False)
ground_truth = pd.read_csv(BASE_TRAIN + 'train_ground_truth.tsv', sep='\t', dtype=str, keep_default_na=False, nrows=1_000)

# Rebuild true_pairs
true_pairs = set()
for row in ground_truth.itertuples(index=False):
    for matched_id in row.matched_entity_ids.split(","):
        matched_id = matched_id.strip()
        if matched_id:
            true_pairs.add((row.source1_entity_id, matched_id))

print(f"Ground truth rows loaded : {len(ground_truth):,}")
print(f"Total true pairs         : {len(true_pairs):,}")  # expect ~3,000-4,000

# Filter source1 to only the 1000 rows used
s1_ids = set(candidate_details['source1_entity_id'])
source1 = source1[source1['entity_id'].isin(s1_ids)]

print(f"candidate_details rows: {len(candidate_details):,}")
print(f"source1 rows: {len(source1):,}")
print(f"true pairs: {len(true_pairs):,}")

# Get only the candidate S2/S3 ids we actually generated
candidate_ids = set(candidate_details['candidate_entity_id'])

# Read S2 and S3 but only keep rows we need
def read_filtered(path, ids):
    chunks = []
    for chunk in pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False, chunksize=250_000):
        filtered = chunk[chunk['entity_id'].isin(ids)]
        if not filtered.empty:
            chunks.append(filtered)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

s2_filtered = read_filtered(SOURCE2_PATH, candidate_ids)
s3_filtered = read_filtered(SOURCE3_PATH, candidate_ids)
s2_s3_filtered = pd.concat([s2_filtered, s3_filtered], ignore_index=True)

print(f"Candidate S2/S3 records fetched: {len(s2_s3_filtered):,}")

enriched = (
    candidate_details
    .merge(
        source1[['entity_id', 'business_name', 'business_address', 'country']].rename(columns={
            'entity_id':        's1_entity_id',
            'business_name':    's1_name',
            'business_address': 's1_address',
            'country':          's1_country',
        }),
        left_on='source1_entity_id',
        right_on='s1_entity_id',
        how='left'
    )
    .merge(
        s2_s3_filtered[['entity_id', 'business_name', 'business_address', 'country']].rename(columns={
            'entity_id':        'cand_entity_id',
            'business_name':    'cand_name',
            'business_address': 'cand_address',
            'country':          'cand_country',
        }),
        left_on='candidate_entity_id',
        right_on='cand_entity_id',
        how='left'
    )
    .drop(columns=['s1_entity_id', 'cand_entity_id'])
)

enriched = enriched[[
    'source1_entity_id', 's1_name', 's1_address', 's1_country',
    'candidate_entity_id', 'cand_name', 'cand_address', 'cand_country',
    'channels', 'channel_count'
]]

# Mark true matches
enriched['is_true_match'] = enriched.apply(
    lambda row: (row['source1_entity_id'], row['candidate_entity_id']) in true_pairs,
    axis=1
)

print(f"Enriched pairs: {len(enriched):,}")
print(f"True matches visible: {enriched['is_true_match'].sum():,}")
print(f"False candidates: {(~enriched['is_true_match']).sum():,}")

display(enriched.head(30))

Ground truth rows loaded : 1,000
Total true pairs         : 3,444
candidate_details rows: 448,861
source1 rows: 982
true pairs: 3,444
Candidate S2/S3 records fetched: 201,333
Enriched pairs: 448,861
True matches visible: 282
False candidates: 448,579


,source1_entity_id,s1_name,s1_address,s1_country,candidate_entity_id,cand_name,cand_address,cand_country,channels,channel_count,is_true_match
0,S1-490372842,Physical Therapy Group,"510 Mettabrook Drive, Swanton, OH",US,S2-925511054,Physical Therapy (Group),"1220 MAIN ST, ASHLAND, OH",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
1,S1-745867296,Historical Foundation,"22600 Dewberry Hollow, Leander, TX",US,S2-880764652,Historical Foundation,"2712 WINSTON ROAD, PILOT MOUNTAIN, NC",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
2,S1-897282581,Bowden Ridge Development LLC,"248 Night Riders Way, Jarrell, TX",US,S2-522109781,BOWDEN RIDGE DEVELOPMENT LLC,"248 NIGHT RIDERS WAY, TX, JARRELL",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,True
3,S1-99560436,Ear Nose & Throat Medicine LLC,"703 Beacon Court, Montgomery Township, PA",US,S2-843937862,EAR NOSE & THROAT MEDICINE LLC,,US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
4,S1-518197819,Primary Care Clinic LLC,"351 11th Street, Washington, DC",US,S2-596878321,PRIMARY CARE CLINIC LLC,"102 9RD AVE, CAPITOL, AZ",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
5,S1-148802034,Citadel,"803 Cushing Street, South Bend, IN",US,S2-347549160,Citadel,"00609 WOODLAWN AVENUE, WILMINGTON, NC",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
6,S1-291419333,Cozy Hypnosis!,"121 Friendship Street, Unit 2, Fall River, MA",US,S2-758947477,Cozy Hypnosis,,US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
7,S1-602144510,International Management Private Limited,"C-93, G/F, Ring Road, Ndse-Ii, Delhi, New Delhi, Delhi",India,S2-979515993,International Management [Private-Limited],"HN 218 PRIDE CLASSIC 4TH F SHOP, 408 PATIDAR CHOWK SV ROAD, RAJKOT, ગુજરાત",India,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
8,S1-274126313,"Obsidian, LLC","3907 Hamilton Road, Deer Park, WA",US,S2-99534465,Obsidian LLC,"75-55 184 ST, FRESH MADOWS, NY",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
9,S1-518197819,Primary Care Clinic LLC,"351 11th Street, Washington, DC",US,S2-184654292,Primary Care Clinic LLC,"MA, 390 MAMMOTH RAAD, LOWELL",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False


In [21]:
enriched_s3 = enriched[enriched['candidate_entity_id'].str.startswith('S3-')]

print(f"S3 candidate pairs: {len(enriched_s3):,}")
display(enriched_s3.head(30))

S3 candidate pairs: 240,988


,source1_entity_id,s1_name,s1_address,s1_country,candidate_entity_id,cand_name,cand_address,cand_country,channels,channel_count,is_true_match
207873,S1-935930929,Apex Consultancy Private Limited,"Plot 692, House No 3/191(New), 3/40 (Old) Nandhi Hills, Meerpet, Hyderabad, Telangana",India,S3-891626766,Apex Consultancy Private [Limited],"867 Ground-floor G-1, Domjur, 296/1 G.t.road Belur, Bally, পশ্চিমবঙ্গ",India,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
207874,S1-425128718,A Cure 6 IT,"6942 Talton Drive, Franklin, OH",US,S3-452675423,A Cure 6 IT,"89-21 78 Street, Woodhaven, New York",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
207875,S1-274115163,Blue Agro Private Limited,"32, Dharnidhar Co. Op. Ho. Society, Nr. Vikasgruh, Op. Dharnidhar Derasar, Paldi, Ahmedabad, Gujarat",India,S3-464381537,Blue Agro-Private Limited,"19/772/4, Kollam, കേരളം",India,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
207876,S1-196935438,Ironclad,"7460 300, Greensburg, IN",US,S3-571388277,Ironclad,"Syracuse, 003478 Gray Pine Drive, Utah",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
207877,S1-625272950,Urology Care Associates,"Harlingen, 163 Nueces Park Lane, TX",US,S3-147388550,Urology Care Associates,"7875 Farm Road 34, Missouri, Walnut Grove CDP",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
207878,S1-456318790,Pediatric Dental Center LLC,"1151 Pioneer Drive, Cottonwood, AZ",US,S3-790347559,Pediatric Dental Center LLC,"Brentwood Drive, Indianapols CITY, Indiana",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
207879,S1-142523012,Oncology Medicine LLC,"55 Fulton Street, Hampton City, VA",US,S3-247019154,Oncology Medicine (Llc),"2762 Spout Ln, Lusby, Maryland",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
207880,S1-721911045,Mount Baptist Church,"18170 Golden Valley Drive, Sahuarita, AZ",US,S3-492931656,MOUNT BAPTIST CHURCH,"5013-C Louise Ave, Sioux Falls, South Dakota",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
207881,S1-764841010,Ridge Federation LP,"200 New Ipswich Road, Ashby, MA",US,S3-142391530,Ridge Federation LP,"01683 Dorset Drive, Avon, Indiana",US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,False
207882,S1-568364082,Caque Semiconductor LLC,"351 Oakdale Avenue, Mineral Wells, WV",US,S3-336170976,Caque Semiconductor LLC,,US,"accent_core_name,core_name,exact_name,tfidf_cosine,token_sort_fuzzy,token_sorted",6,True


In [22]:
# Add true match flag
enriched['is_true_match'] = enriched.apply(
    lambda row: (row['source1_entity_id'], row['candidate_entity_id']) in true_pairs,
    axis=1
)

# ── Overall metrics ──────────────────────────────────────────────────────────
total_candidates = len(enriched)
true_matches_retrieved = enriched['is_true_match'].sum()
false_candidates = (~enriched['is_true_match']).sum()
total_true_pairs = len(true_pairs)

precision = true_matches_retrieved / total_candidates if total_candidates else 0
recall    = true_matches_retrieved / total_true_pairs if total_true_pairs else 0
f05       = (1.25 * precision * recall) / (0.25 * precision + recall) if (precision + recall) else 0

print("=" * 50)
print("OVERALL BLOCKING METRICS")
print("=" * 50)
print(f"Total candidate pairs        : {total_candidates:,}")
print(f"True matches retrieved       : {true_matches_retrieved:,}")
print(f"False candidates (noise)     : {false_candidates:,}")
print(f"True pairs in ground truth   : {total_true_pairs:,}")
print(f"True pairs MISSED            : {total_true_pairs - true_matches_retrieved:,}")
print()
print(f"Blocking Precision           : {precision:.2%}")
print(f"Blocking Recall              : {recall:.2%}")
print(f"Blocking F0.5                : {f05:.4f}")
print(f"Reduction Ratio              : {1 - total_candidates / (len(source1) * (len(s2_s3_filtered))):,.4f}")

# ── Per channel breakdown ─────────────────────────────────────────────────────
print()
print("=" * 50)
print("METRICS BY CHANNEL")
print("=" * 50)

all_channels = ['exact_name', 'core_name', 'accent_core_name', 'exact_address']

for ch in all_channels:
    ch_mask     = enriched['channels'].str.contains(ch)
    ch_total    = ch_mask.sum()
    ch_correct  = enriched[ch_mask]['is_true_match'].sum()
    ch_precision = ch_correct / ch_total if ch_total else 0
    print(f"{ch:<22} | candidates: {ch_total:>6,} | true matches: {ch_correct:>5,} | precision: {ch_precision:.2%}")

# ── Per source breakdown ──────────────────────────────────────────────────────
print()
print("=" * 50)
print("METRICS BY SOURCE (S2 vs S3)")
print("=" * 50)

for src in ['S2', 'S3']:
    src_df      = enriched[enriched['candidate_entity_id'].str.startswith(src + '-')]
    src_total   = len(src_df)
    src_correct = src_df['is_true_match'].sum()
    src_prec    = src_correct / src_total if src_total else 0
    # recall for this source only
    src_true    = sum(1 for _, cid in true_pairs if cid.startswith(src + '-'))
    src_recall  = src_correct / src_true if src_true else 0
    print(f"{src} | candidates: {src_total:>6,} | true: {src_correct:>5,} | precision: {src_prec:.2%} | recall: {src_recall:.2%}")

# ── Channel count breakdown ───────────────────────────────────────────────────
print()
print("=" * 50)
print("PRECISION BY NUMBER OF CHANNELS THAT FIRED")
print("=" * 50)

enriched['channel_count'] = enriched['channel_count'].astype(int)
for n in sorted(enriched['channel_count'].unique()):
    n_df   = enriched[enriched['channel_count'] == n]
    n_prec = n_df['is_true_match'].mean()
    print(f"{n} channel(s) fired | candidates: {len(n_df):>6,} | precision: {n_prec:.2%}")

# ── Missed true pairs ─────────────────────────────────────────────────────────
print()
print("=" * 50)
print("SAMPLE OF MISSED TRUE PAIRS (not retrieved by blocking)")
print("=" * 50)

retrieved_pairs = set(zip(enriched['source1_entity_id'], enriched['candidate_entity_id']))
missed_pairs    = true_pairs - retrieved_pairs

print(f"Total missed: {len(missed_pairs):,}")

OVERALL BLOCKING METRICS
Total candidate pairs        : 448,861
True matches retrieved       : 282
False candidates (noise)     : 448,579
True pairs in ground truth   : 3,444
True pairs MISSED            : 3,162

Blocking Precision           : 0.06%
Blocking Recall              : 8.19%
Blocking F0.5                : 0.0008
Reduction Ratio              : 0.9977

METRICS BY CHANNEL
exact_name             | candidates:    934 | true matches:    76 | precision: 8.14%
core_name              | candidates:  3,877 | true matches:   158 | precision: 4.08%
accent_core_name       | candidates:  3,877 | true matches:   158 | precision: 4.08%
exact_address          | candidates:     29 | true matches:    21 | precision: 72.41%

METRICS BY SOURCE (S2 vs S3)
S2 | candidates: 207,873 | true:   143 | precision: 0.07% | recall: 8.63%
S3 | candidates: 240,988 | true:   139 | precision: 0.06% | recall: 7.78%

PRECISION BY NUMBER OF CHANNELS THAT FIRED
1 channel(s) fired | candidates: 306,679 | precision: 

In [23]:
# Get the actual records for missed pairs to understand WHY blocking failed
missed_pairs = true_pairs - retrieved_pairs

# Convert to dataframe
missed_df = pd.DataFrame(list(missed_pairs), columns=['source1_entity_id', 'candidate_entity_id'])

# Get candidate ids that were missed
missed_candidate_ids = set(missed_df['candidate_entity_id'])

# Fetch the actual S2/S3 records for missed candidates
def read_filtered(path, ids):
    chunks = []
    for chunk in pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False, chunksize=250_000):
        filtered = chunk[chunk['entity_id'].isin(ids)]
        if not filtered.empty:
            chunks.append(filtered)
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()

missed_s2 = read_filtered(SOURCE2_PATH, missed_candidate_ids)
missed_s3 = read_filtered(SOURCE3_PATH, missed_candidate_ids)
missed_cands = pd.concat([missed_s2, missed_s3], ignore_index=True)

# Join with S1 to see both sides
missed_enriched = (
    missed_df
    .merge(
        source1[['entity_id', 'business_name', 'business_address', 'country']].rename(columns={
            'entity_id': 's1_id', 'business_name': 's1_name',
            'business_address': 's1_address', 'country': 's1_country'
        }),
        left_on='source1_entity_id', right_on='s1_id', how='left'
    )
    .merge(
        missed_cands[['entity_id', 'business_name', 'business_address', 'country']].rename(columns={
            'entity_id': 'cand_id', 'business_name': 'cand_name',
            'business_address': 'cand_address', 'country': 'cand_country'
        }),
        left_on='candidate_entity_id', right_on='cand_id', how='left'
    )
    .drop(columns=['s1_id', 'cand_id'])
)

print(f"Missed pairs: {len(missed_enriched):,}")
print(f"\nBy source:")
print(missed_enriched['candidate_entity_id'].str[:2].value_counts())
print(f"\nBy country:")
print(missed_enriched['s1_country'].value_counts())

display(missed_enriched.head(30))

Missed pairs: 3,162

By source:
candidate_entity_id
S3    1648
S2    1514
Name: count, dtype: int64

By country:
s1_country
US       1878
India    1229
Name: count, dtype: int64


,source1_entity_id,candidate_entity_id,s1_name,s1_address,s1_country,cand_name,cand_address,cand_country
0,S1-96602127,S2-848899456,GVS Metal Private Limited,"8Th Km Stone Nagina Roadnajibabad Bijnor, Uttar Pradesh, Bijnore, Uttar Pradesh",India,GVS Metal Pvaet Limited,"8TH KM STONE NAGINA ROADNAJIBABAD BIJNOR, UTTAR PRADESH, Uttar Pradesh",India
1,S1-117022003,S3-381651665,Paque Family Office,"413 Sixth Street, WI, Village Of Waterford",US,Paque Family Office,"Waterford, Wisconsin, 415 6th Street",US
2,S1-731397769,S2-962774422,Fasoft Midwest Biopharma,"112 Hummingbird Hill Road, Franklin, NC",US,Fasoft Midwest,"112 HUMMINGBIRD HILL RD, FRANKLIN, NC",US
3,S1-61434219,S2-521757954,Hitech Engineering Private Limited,"Shop 401 In 2214 Forth Floor, Gurudwara Road, Karol Bagh, Delhi, Central Delhi, Delhi",India,हाईटेक इंजीनियरिंग प्राइवेट लिमिटेड,"PLOT 577 SHOP 401 IN 2214 FORTH FLOOR, GURUDWARA ROAD, KAROL BAGH, CENTRAL, CENTRAL DELHI, Delhi",India
4,S1-798312676,S3-256208237,Vision Impex Private Limited,"Flat No - 003, B-23, R N B Bldg, Anand Nagar, Dahisar East, Mumbai, Maharashtra",India,Lumarc,"Flat No - 003, Dahisar East, Mumbai, MH",India
5,S1-681402684,S3-221909416,Big 15 Bakery,"8220 Sunday Drive, Henrico County, VA",US,Big 15 Bakery,"Sunday Drive, Henrico County, Virginia",US
6,S1-9357479,S3-723246327,Jhansi India Private Limited,"House No. 1527 C3, Near Janki Puram Civil Lines, Jhansi, Uttar Pradesh",India,Jhansi India Private Ltd,"House No. 1527. C3, Near Janki Puram Civil Lines, Jhansi, Barwasagar, UP",India
7,S1-618676054,S2-248107822,Swanson Golden Foods Corp,"106 Sundew Lane, Cary, NC",US,Swanson Golden Foods Corp.,"106 SUNDEW LN, CARY, NC",US
8,S1-605002576,S2-306446692,Commit Homes (India) Pvt Ltd,"At- Baitpara, Balasore, Orissa, Basta, Balasore, C/O Pradeep Hota",India,Commit Homes (India) Pvt,"C/O PRADEEP HOTA, AT- BAITPARA, BALASORE, BASTA, ଓଡ଼ିଶା",India
9,S1-692219144,S2-523089825,Cornerstone Desert Enbridge,"6613 Edith Boulevard, Albuquerque, NM",US,CORNERSTONE DESERT ENBRIDGE,"Edith Blvd, ALBUQUERQUE, NM",US
